In [ ]:
!pip install "transformers<5" -q

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "tomg-group-umd/huginn-0125",
    trust_remote_code=True,
    device_map="auto",
    dtype=torch.bfloat16 # Updated from torch_dtype to dtype
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
n = 20          # number of examples to run
num_steps = 32  # Huginn's recurrent depth (more = more test-time compute)

In [ ]:
import re
from datasets import load_dataset
from transformers import AutoTokenizer, GenerationConfig

tokenizer = AutoTokenizer.from_pretrained("tomg-group-umd/huginn-0125")

# Load GSM8K and keep only the first n examples
ds = load_dataset("openai/gsm8k", "main", split="test").select(range(n))

gen_config = GenerationConfig(
    max_new_tokens=512,
    stop_strings=["<|end_text|>", "<|end_turn|>"],
    use_cache=True,
    do_sample=False,
    temperature=None, top_k=None, top_p=None, min_p=None,
    return_dict_in_generate=True,
    eos_token_id=65505,
    bos_token_id=65504,
    pad_token_id=65509,
)

In [ ]:
def extract_number(text):
    """Return the last number in a string, stripped of commas/$."""
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return nums[-1].replace(",", "").rstrip(".") if nums else None

In [ ]:
def is_correct(pred, gold):
    try:
        return pred is not None and float(pred) == float(gold)
    except ValueError:
        return False

In [ ]:
results = []
for i, ex in enumerate(ds):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": ex["question"] + "\nThink step by step, then give the final answer as a number."},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    out = model.generate(input_ids, gen_config, tokenizer=tokenizer, num_steps=num_steps)
    new_tokens = out.sequences[0, input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    pred = extract_number(response)
    correct = is_correct(pred, gold)
    results.append(correct)

    print("=" * 80)
    print(f"Example {i + 1}/{n}")
    print(f"INPUT:\n{ex['question']}\n")
    print(f"RECURRENT STEPS: {num_steps} per token "
          f"({len(new_tokens)} tokens generated, ~{num_steps * len(new_tokens)} core-block passes total)")
    print(f"OUTPUT:\n{response}\n")
    print(f"GOLD: {gold} | PRED: {pred} | {'✓ CORRECT' if correct else '✗ WRONG'}")

print("=" * 80)
print(f"Accuracy: {sum(results)}/{len(results)} = {sum(results) / len(results):.1%}")

Example 1/20
INPUT:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

RECURRENT STEPS: 32 per token (174 tokens generated, ~5568 core-block passes total)
OUTPUT:
To calculate Janet's daily earnings from selling fresh duck eggs at the farmers' market, we need to follow these steps:
1. Determine the total number of eggs laid by the ducks each day.
2. Subtract the eggs eaten for breakfast and baked muffins.
3. Subtract the eggs sold at the market.
4. Calculate the remaining number of eggs.
5. Multiply the remaining number of eggs by the market price per egg.
Step 1: 16 eggs per day
Step 2: 3 eggs eaten for breakfast + 4 eggs baked muffins = 7 eggs eaten
Step 3: 16 eggs - 7 eggs = 9 eggs remaining
Step 4: 9 eggs
Step 5: 9 eggs x $2 per egg = $18
Therefore, Janet

In [ ]:
# Github repo
# Rerun with 20q x {16,8,4} recurrent steps
# Per-question timing %timeit

In [1]:
import timeit


### Methods for Timing Commands in Python & Colab

Below are three common ways to measure the execution time of your code.

In [2]:
import time
import timeit

# --- Method 1: Using the `time` module (Simple & Practical) ---
print("--- Method 1: time.perf_counter() ---")
start_time = time.perf_counter()

# Code block to time
total = sum(i * i for i in range(1000000))

end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.6f} seconds\n")


# --- Method 2: Using the `timeit` module (Accurate Benchmarking) ---
# Useful for running a snippet thousands of times to get an average
print("--- Method 2: timeit.timeit() ---")
execution_time = timeit.timeit(
    stmt="sum(i * i for i in range(1000))",
    number=1000
)
print(f"Execution time for 1000 runs: {execution_time:.6f} seconds\n")


# --- Method 3: Colab/IPython Line and Cell Magics ---
# Note: You can use `%time` or `%timeit` at the start of any line,
# or `%%time` / `%%timeit` at the very top of a cell to time the whole cell.

--- Method 1: time.perf_counter() ---
Elapsed time: 0.182146 seconds

--- Method 2: timeit.timeit() ---
Execution time for 1000 runs: 0.137761 seconds



In [3]:
# Timing a single line using IPython magic
print("--- Method 3a: %time magic (Single execution) ---")
%time total = sum(i * i for i in range(1000000))

print("\n--- Method 3b: %timeit magic (Multiple runs for benchmark) ---")
%timeit total = sum(i * i for i in range(1000))

--- Method 3a: %time magic (Single execution) ---
CPU times: user 152 ms, sys: 215 µs, total: 152 ms
Wall time: 358 ms

--- Method 3b: %timeit magic (Multiple runs for benchmark) ---
247 µs ± 118 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [4]:
results

NameError: name 'results' is not defined

In [5]:
!pwd

/content
